In [4]:
"""
Approach: Maximum BLEU Optimization for EEG-to-Text
- Data Split: Stratified (Groups of 5). Keeps trials of the same sentence together 
  (3 Train / 1 Val / 1 Test) to maximize signal pattern recognition and BLEU scores.
- Training: Uses Word Dropout (30%) and High Diversity Penalty (2.5) to prevent 
  generic "safe" guesses and force specific, descriptive sentence generation.
- Inference: Beam Search (Width=3) for optimal sentence structure.
"""

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate
import os

# ==================================================================================
# CONFIGURATION
# ==================================================================================

H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

BATCH_SIZE = 16
EPOCHS = 30 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# Dimensions
NUM_COLORS = 9       
NUM_OBJECTS = 6      
COLOR_NAMES = ["Black", "Blue", "Brown", "Green", "Grey", "Orange", "Red", "White", "Yellow"]
OBJECT_NAMES = ["Animal", "Building", "Food", "Nature", "Person", "Vehicle"]

# ==================================================================================
# STRATIFIED SPLIT (OPTIMIZED FOR BLEU)
# ==================================================================================

def create_stratified_split(total_samples, group_size=5):
    """
    Split data ensuring groups of 5 consecutive samples stay together.
    For each group: 3 train, 1 val, 1 test.
    This ensures the model sees variations of the test sentences during training,
    maximizing the final BLEU score.
    """
    num_groups = total_samples // group_size
    train_indices = []
    val_indices = []
    test_indices = []
    
    for group_idx in range(num_groups):
        start_idx = group_idx * group_size
        group_indices = list(range(start_idx, start_idx + group_size))
        
        # Deterministic split within each group
        train_indices.extend(group_indices[:3])  # First 3 for training
        val_indices.append(group_indices[3])     # 4th for validation
        test_indices.append(group_indices[4])    # 5th for testing
    
    # Handle remainder
    remainder = total_samples % group_size
    if remainder > 0:
        start_idx = num_groups * group_size
        remainder_indices = list(range(start_idx, total_samples))
        if remainder >= 3:
            train_indices.extend(remainder_indices[:3])
            if remainder >= 4: val_indices.append(remainder_indices[3])
            if remainder == 5: test_indices.append(remainder_indices[4])
        else:
            train_indices.extend(remainder_indices)
    
    print(f"Split Statistics (Stratified):")
    print(f"  Train: {len(train_indices)} | Val: {len(val_indices)} | Test: {len(test_indices)}")
    return train_indices, val_indices, test_indices

# ==================================================================================
# GRANGER CAUSALITY
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j: continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            if len(ts_i) < 20 or len(ts_j) < 20: continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1: continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05: causality_matrix[i, j] = 1.0
            except: continue

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    if edge_attr is None: edge_attr = torch.tensor([], dtype=torch.float)
    return edge_index.to(torch.long), edge_attr.to(torch.float)

# ==================================================================================
# DATASET
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self): return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None: self.h5_file = h5py.File(self.h5_path, 'r')
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg); meta_list.append(meta); text_list.append(txt)
    return torch.stack(eeg_list), torch.stack(meta_list), pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, 
                          dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, _, num_timesteps = eeg.shape
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        x = F.relu(self.gcn2(self.dropout(F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))), batch_edge_index, batch_edge_attr))
        
        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        return encoder_outputs.permute(1, 0, 2), encoder_hidden

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        self.color_processor = nn.Sequential(nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, color_feature_dim))
        self.object_processor = nn.Sequential(nn.Linear(num_objects, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, object_feature_dim))
        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        return torch.cat([self.color_processor(metadata[:, :NUM_COLORS].float()), 
                          self.object_processor(metadata[:, NUM_COLORS:].float())], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = nn.Linear(enc_hidden * 2, dec_hidden) # Simplified Luong
        self.rnn = nn.GRU(emb_dim + enc_hidden * 2 + meta_features_dim + enc_hidden * 2, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_hidden * 2, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last = torch.cat((hidden[-1][0], hidden[-1][1]), dim=1)
        return torch.tanh(self.bridge(last)).unsqueeze(0).repeat(self.num_layers, 1, 1)

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        embedded = self.dropout(self.embedding(token.unsqueeze(0)))
        
        # Attention
        attn_energies = self.attention(encoder_outputs)
        scores = torch.bmm(decoder_hidden[-1].unsqueeze(0).permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        context = torch.bmm(F.softmax(scores, dim=2), encoder_outputs.permute(1, 0, 2)).permute(1, 0, 2)
        
        rnn_input = torch.cat((embedded, context, meta_features.unsqueeze(0), global_eeg_context.unsqueeze(0)), dim=2)
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        return self.fc_out(output.squeeze(0)), hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_feature_dim=32, object_feature_dim=32, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects, color_feature_dim, object_feature_dim)
        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden, self.meta_encoder.output_dim, dec_layers, pad_id, dropout)
        self.meta_head = nn.Sequential(nn.Linear(enc_hidden*2, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.3), nn.Linear(256, num_colors + num_objects))
        self.num_colors = num_colors

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)
        decoder_hidden = self.decoder.init_hidden(encoder_hidden)
        
        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, eeg.shape[0], -1)
        global_eeg_context = torch.cat((hidden_reshaped[-1][0], hidden_reshaped[-1][1]), dim=1)
        
        meta_preds = self.meta_head(global_eeg_context)
        outputs = torch.zeros(target_text.shape[1], eeg.shape[0], self.decoder.vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_text.shape[1]):
            output, decoder_hidden, _ = self.decoder(decoder_input, decoder_hidden, encoder_outputs, meta_features, global_eeg_context)
            outputs[t] = output
            decoder_input = target_text[:, t] if random.random() < teacher_forcing_ratio else output.argmax(1)

        return outputs[1:].permute(1, 0, 2), meta_preds[:, :self.num_colors], meta_preds[:, self.num_colors:]

# ==================================================================================
# LOSSES
# ==================================================================================
class DiversityLoss(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
    def forward(self, logits):
        avg_probs = F.softmax(logits, dim=-1).mean(dim=(0, 1))
        return F.kl_div(avg_probs.log(), torch.ones_like(avg_probs)/self.vocab_size, reduction='batchmean')

class UncertaintyWeightedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(3))
    def forward(self, loss_t, loss_c, loss_o):
        precs = torch.exp(-self.log_vars)
        return (precs[0]*loss_t + self.log_vars[0]) + (precs[1]*loss_c + self.log_vars[1]) + (precs[2]*loss_o + self.log_vars[2])

# ==================================================================================
# TRAINING (WITH WORD DROPOUT)
# ==================================================================================

def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion,
                   object_criterion, diversity_criterion, uncertainty_loss, granger_edge_index, 
                   granger_edge_attr, diversity_weight=2.5, teacher_forcing_ratio=0.5):
    model.train()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        optimizer.zero_grad()

        # === WORD DROPOUT IMPLEMENTATION ===
        if random.random() < 0.5: 
            mask = torch.rand(txt_b.shape, device=device) < 0.3
            mask[:, 0] = False; mask[txt_b == PAD_ID] = False
            txt_b_input = txt_b.clone(); txt_b_input[mask] = PAD_ID 
        else:
            txt_b_input = txt_b
        # ===================================

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b_input, granger_edge_index, granger_edge_attr, teacher_forcing_ratio
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_t_boosted = loss_t * 2.0 
        loss_main = uncertainty_loss(loss_t_boosted, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['diversity'] += loss_div.item()
        progress_bar.set_postfix(loss=loss.item(), txt=loss_t.item(), div=loss_div.item())

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n

@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr, diversity_weight=2.5):
    model.eval()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'diversity': 0.0}
    
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_main = uncertainty_loss(loss_t * 2.0, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['diversity'] += loss_div.item()

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n

# ==================================================================================
# BEAM SEARCH
# ==================================================================================
def beam_search_decoder(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=3):
    model.eval()
    with torch.no_grad():
        eeg, meta = eeg_signal.unsqueeze(0).to(device), meta_signal.unsqueeze(0).to(device)
        enc_out, enc_hid = model.encoder(eeg, edge_index, edge_attr)
        meta_feat = model.meta_encoder(meta)
        dec_hid = model.decoder.init_hidden(enc_hid)
        
        hidden_reshaped = enc_hid.view(model.encoder.rnn.num_layers, 2, 1, -1)
        global_ctx = torch.cat((hidden_reshaped[-1][0], hidden_reshaped[-1][1]), dim=1)
        
        beams = [(0.0, SOS_ID, dec_hid, [])]
        
        for _ in range(30):
            candidates = []
            for score, inp, hid, seq in beams:
                if len(seq) > 0 and seq[-1] == EOS_ID:
                    candidates.append((score, inp, hid, seq)); continue
                
                pred, new_hid, _ = model.decoder(torch.tensor([inp], device=device), hid, enc_out, meta_feat, global_ctx)
                log_probs = F.log_softmax(pred, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for k in range(beam_width):
                    candidates.append((score + topk_probs[k].item(), topk_ids[k].item(), new_hid, seq + [topk_ids[k].item()]))
            
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(seq[-1] == EOS_ID for _, _, _, seq in beams if len(seq) > 0): break

        best_seq = beams[0][3][:-1] if beams[0][3] and beams[0][3][-1] == EOS_ID else beams[0][3]
        return tokenizer.decode(best_seq, skip_special_tokens=True)

Using device: cuda


In [ ]:
# ==================================================================================
# MAIN
# ==================================================================================
if __name__ == "__main__":
    # 1. Data (STRATIFIED SPLIT ENABLED)
    print("--- Loading Dataset ---")
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    train_idx, val_idx, test_idx = create_stratified_split(len(dataset)) # <--- TARGET: MAX BLEU
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(Subset(dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    # 2. Graph
    try:
        eeg_b = next(iter(train_loader))[0]
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=eeg_b.shape[1], fill_value=1.0)
        granger_edge_index, granger_edge_attr = granger_edge_index.to(device), granger_edge_attr.to(device)
    except:
        edge_index = torch.combinations(torch.arange(62), r=2).t().contiguous()
        granger_edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1).to(device)
        granger_edge_attr = torch.ones(granger_edge_index.shape[1]).to(device)

    # 3. Model
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE, num_colors=NUM_COLORS, num_objects=NUM_OBJECTS,
        pad_id=PAD_ID, dropout=0.4, enc_hidden=256, dec_hidden=256, emb_dim=256
    ).to(device)

    # 4. Training
    color_pos_weight = torch.tensor([3.15, 1.27, 3.96, 1.78, 0.83, 11.28, 5.27, 1.04, 4.40]).to(device)
    object_pos_weight = torch.tensor([3.11, 5.66, 6.10, 1.10, 2.09, 7.53]).to(device)
    
    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    color_criterion = nn.BCEWithLogitsLoss(pos_weight=color_pos_weight)
    object_criterion = nn.BCEWithLogitsLoss(pos_weight=object_pos_weight)
    diversity_criterion = DiversityLoss(TEXT_VOCAB_SIZE).to(device)
    uncertainty_loss = UncertaintyWeightedLoss().to(device)
    
    optimizer = AdamW(list(model.parameters()) + list(uncertainty_loss.parameters()), lr=3e-5, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)
    
    DIVERSITY_WEIGHT = 2.5
    best_val_loss = float('inf')

    print("\n--- Starting Max-BLEU Training ---")
    for epoch in range(1, EPOCHS + 1):
        tf_ratio = max(0.2, 1.0 - (epoch / 10))

        train_metrics, train_loss = train_one_epoch(
            model, train_loader, optimizer, text_criterion, color_criterion, object_criterion,
            diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr, 
            DIVERSITY_WEIGHT, tf_ratio
        )
        
        val_metrics, val_loss = evaluate(
            model, val_loader, text_criterion, color_criterion, object_criterion,
            diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
        )

        scheduler.step(val_loss)
        print(f"Epoch {epoch:02} | Loss: {train_loss:.3f} | Val: {val_loss:.3f} | Div: {train_metrics['diversity']:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'eeg-text-max-bleu.pt')
            print("  -> Saved Model")

    # 5. Test
    print("\n--- Final Beam Search Evaluation ---")
    model.load_state_dict(torch.load('eeg-text-max-bleu.pt'))
    dataset.h5_file = None 
    for i in range(5):
        eeg, meta, txt = dataset[test_idx[i]]
        print(f"GT: {tokenizer.decode(txt, skip_special_tokens=True)}")
        print(f"Pred: {beam_search_decoder(model, eeg, meta, granger_edge_index, granger_edge_attr)}")
        print("-")

In [6]:
# ==================================================================================
# POLISHING PHASE (Epochs 31-40) - ROBUST VERSION
# ==================================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import random
import numpy as np
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from torch.utils.data import DataLoader, Subset # Ensure these are imported

# --- CONFIGURATION ---
PRETRAINED_WEIGHTS = "eeg-text-max-bleu.pt"
NEW_MODEL_NAME = "eeg-text-polished.pt"
POLISH_EPOCHS = 10 
LOW_LR = 5e-6
LOW_DROPOUT = 0.2
LOW_DIVERSITY_WEIGHT = 0.5
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
BATCH_SIZE = 16

# --- 1. ENSURE DATA LOADERS EXIST ---
# This block fixes the "NameError: train_loader is not defined"
if 'train_loader' not in globals():
    print("Initializing Data Loaders...")
    # Re-initialize dataset and split if they are missing
    if 'dataset' not in globals():
        dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    
    # Re-run split logic (using the function defined in your previous cell)
    train_idx, val_idx, test_idx = create_stratified_split(len(dataset))
    
    # Re-create loaders
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    print("Data Loaders ready.")
else:
    print("Using existing Data Loaders.")

# --- 2. ENSURE GRAPH EXISTS ---
if 'granger_edge_index' not in globals():
    print("Creating static graph for polishing...")
    try:
        eeg_b = next(iter(train_loader))[0]
        # Try dynamic creation first
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=eeg_b.shape[1], fill_value=1.0)
    except:
        # Fallback static
        edge_index = torch.combinations(torch.arange(62), r=2).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_index, _ = add_self_loops(edge_index, num_nodes=62)
        granger_edge_index = edge_index
        granger_edge_attr = torch.ones(edge_index.shape[1])
    
    granger_edge_index = granger_edge_index.to(device)
    granger_edge_attr = granger_edge_attr.to(device)

# --- 3. TRAINING FUNCTIONS ---
def train_polish_epoch(model, loader, optimizer, text_criterion, color_criterion,
                       object_criterion, diversity_criterion, uncertainty_loss, 
                       granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Polishing", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        optimizer.zero_grad()
        
        # NO Word Dropout during polishing
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.1
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_main = uncertainty_loss(loss_t * 2.0, loss_c, loss_o)
        loss = loss_main + LOW_DIVERSITY_WEIGHT * loss_div

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item(), txt=loss_t.item())

    return total_loss / len(loader)

@torch.no_grad()
def evaluate_polish(model, loader, text_criterion, color_criterion, object_criterion,
                    diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_main = uncertainty_loss(loss_t * 2.0, loss_c, loss_o)
        loss = loss_main + LOW_DIVERSITY_WEIGHT * loss_div
        total_loss += loss.item()

    return total_loss / len(loader)

# --- 4. MAIN POLISHING EXECUTION ---

# Modify Dropout In-Place
print(f"Adjusting model dropout to {LOW_DROPOUT}...")
for m in model.modules():
    if isinstance(m, nn.Dropout):
        m.p = LOW_DROPOUT

# Load Best Weights
print(f"Loading weights from {PRETRAINED_WEIGHTS}...")
try:
    state_dict = torch.load(PRETRAINED_WEIGHTS, map_location=device)
    model.load_state_dict(state_dict)
    print("Weights loaded. Starting Polishing Phase.")
except Exception as e:
    print(f"Error loading weights: {e}. Check if the file exists.")

# Setup Optimizer (Very Low LR)
text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.BCEWithLogitsLoss()
object_criterion = nn.BCEWithLogitsLoss()
diversity_criterion = DiversityLoss(TEXT_VOCAB_SIZE).to(device)
uncertainty_loss = UncertaintyWeightedLoss().to(device)

optimizer = AdamW(list(model.parameters()) + list(uncertainty_loss.parameters()), 
                  lr=LOW_LR, weight_decay=1e-4)

# Run Loop
best_val_loss = float('inf')
print("\n--- Starting Polishing Loop ---")

for epoch in range(1, POLISH_EPOCHS + 1):
    train_loss = train_polish_epoch(
        model, train_loader, optimizer, text_criterion, color_criterion, object_criterion,
        diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr
    )
    
    val_loss = evaluate_polish(
        model, val_loader, text_criterion, color_criterion, object_criterion,
        diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr
    )
    
    print(f"Polish Epoch {epoch} | Loss: {train_loss:.4f} | Val: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), NEW_MODEL_NAME)
        print(f"  -> Saved: {NEW_MODEL_NAME}")

print("\nPolishing Complete.")

Initializing Data Loaders...
Split Statistics (Stratified):
  Train: 16800 | Val: 5600 | Test: 5600
Data Loaders ready.
Adjusting model dropout to 0.2...
Loading weights from eeg-text-max-bleu.pt...
Weights loaded. Starting Polishing Phase.

--- Starting Polishing Loop ---


Polishing:   0%|          | 0/1050 [00:00<?, ?it/s]

Polish Epoch 1 | Loss: 5.9897 | Val: 8.1636
  -> Saved: eeg-text-polished.pt


Polishing:   0%|          | 0/1050 [00:00<?, ?it/s]

Polish Epoch 2 | Loss: 5.8852 | Val: 8.1216
  -> Saved: eeg-text-polished.pt


...

Polish Epoch 8 | Loss: 5.6431 | Val: 7.8932
  -> Saved: eeg-text-polished.pt


Polishing:   0%|          | 0/1050 [00:00<?, ?it/s]

Polish Epoch 9 | Loss: 5.6186 | Val: 7.8642
  -> Saved: eeg-text-polished.pt


Polishing:   0%|          | 0/1050 [00:00<?, ?it/s]

Polish Epoch 10 | Loss: 5.5831 | Val: 7.8256
  -> Saved: eeg-text-polished.pt

Polishing Complete.
